# 01. 프롬프트 엔지니어링 실습

---

## 학습 목표

LLM의 출력 품질을 높이는 **핵심 프롬프트 기법 6가지**를 실습합니다.  
같은 질문에 다른 기법을 적용해 결과 차이를 직접 확인합니다.

## 다루는 기법

| 기법 | 핵심 아이디어 | 언제 쓰나? |
|---|---|---|
| **Zero-shot** | 예시 없이 바로 질문 | 간단한 일반 질문 |
| **Few-shot** | 예시(샷)를 프롬프트에 포함 | 특정 형식·스타일 유도 |
| **Chain-of-Thought** | 단계별 추론 유도 | 복잡한 분석·판단 |
| **역할 프롬프트** | LLM에게 페르소나 부여 | 전문 도메인 답변 |
| **구조화된 출력** | 특정 형식(JSON 등) 강제 | 파싱이 필요한 경우 |
| **프롬프트 템플릿** | 변수로 재사용 가능한 프롬프트 | 반복 작업 자동화 |

## 흐름

```
기법 학습 → 코드 실습 → 같은 질문으로 비교 실험 → 핵심 정리
```

In [1]:
# 상위폴더 path 설정
import sys; sys.path.append("..")
from common_config import llm_connect
llm = llm_connect("gpt-5.4-mini")

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

import json

# LLM 초기화
# llm: 환경 설정 셀에서 만든 llm_connect 객체 사용

print("준비 완료")
print("실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'")

준비 완료
실습에 사용할 공통 질문: '재택근무 전면 도입, 찬성해야 할까요?'


---

## 1. Zero-shot Prompting

**예시 없이** 바로 질문하는 가장 기본적인 방식입니다.  
LLM이 사전 학습 지식만으로 답변합니다.

```
사용자: [질문]
LLM:   [답변]
```

In [3]:
# ── Zero-shot: 예시 없이 바로 질문 ────────────────────────────
zero_shot_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = zero_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Zero-shot 결과 ===")
print(result)

=== Zero-shot 결과 ===
결론부터 말하면, **“전면 도입”에 무조건 찬성하기보다는, 업무 특성에 맞는 하이브리드/선택형 모델에 찬성하는 쪽이 더 현실적**입니다.  
다만 **완전 재택이 특히 잘 맞는 조직**이라면 찬성 근거도 충분합니다.

## 찬성할 수 있는 이유
- **생산성 향상 가능성**: 이동 시간 감소, 집중 업무 환경 확보
- **인재 확보에 유리**: 지역 제약 없이 채용 가능
- **비용 절감**: 사무실 임대, 운영비 축소
- **일·생활 균형 개선**: 직원 만족도, 이직률 개선 가능
- **재난/감염병 대응력 강화**: 업무 연속성 확보

## 반대 또는 신중해야 하는 이유
- **협업·소통 저하**: 즉시 피드백, 비공식 소통이 약해질 수 있음
- **신입 교육/온보딩 어려움**: 배움이 느려질 수 있음
- **조직문화 약화**: 소속감, 팀 결속이 떨어질 수 있음
- **성과 관리가 어려움**: 관리 방식이 바뀌지 않으면 비효율 발생
- **업무별 적합성 차이**: 대면이 필요한 업무는 품질 저하 가능

## 그래서 어떻게 보는 게 좋나
- **개인 입장**: 본인 업무가 문서·개발·디자인·분석 중심이면 찬성 가능성이 큼
- **관리자/조직 입장**: 전면보다는  
  **“주 2~3일 재택 + 대면 협업일 지정”** 같은 방식이 더 안정적
- **핵심은 제도보다 운영**: 목표 설정, 커뮤니케이션 규칙, 평가 방식이 갖춰져야 성공함

## 한 줄 정리
**“재택근무 전면 도입”은 업종과 조직문화에 따라 다르며, 일반적으로는 전면 찬성보다 유연한 하이브리드가 더 설득력 있습니다.**

원하시면 제가 이 주제로 **찬성/반대 논설문 형태**, **면접 답변용


---

## 2. Few-shot Prompting

프롬프트 안에 **예시(shots)** 를 포함해서 원하는 출력 형식을 보여줍니다.  
LLM이 패턴을 학습해 같은 형식으로 답변합니다.

```
시스템: 예시1: Q→A, 예시2: Q→A  ← 패턴 학습
사용자: [새 질문]
LLM:   [예시와 같은 형식의 답변]  ← 패턴 따라함
```

In [4]:
# ── Few-shot: 예시 2개를 포함해서 출력 형식을 유도 ────────────
few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 예시와 똑같은 형식으로 답변하세요.

예시 1)
질문: 주 4일 근무제를 도입해야 할까요?
답변:
✅ 찬성 근거
- 직원 삶의 질 향상으로 생산성 증가
- 채용 경쟁력 강화

❌ 반대 근거
- 업무량 미감소 시 오히려 번아웃 위험
- 서비스 연속성 유지 어려움

💡 결론: 업종·직무별로 차등 적용하는 파일럿 테스트를 권장합니다.

예시 2)
질문: 사내 카페테리아를 없애야 할까요?
답변:
✅ 찬성 근거
- 운영 비용 절감
- 외부 다양한 식사 옵션 활용 가능

❌ 반대 근거
- 직원 식사 복지 감소
- 점심 시간 이탈로 협업 기회 감소

💡 결론: 직원 수요 조사 후 결정하되, 식대 지원으로 보완을 고려하세요."""),
    ("human", "질문: {question}")
])

# 체인 생성 및 실행
chain = few_shot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Few-shot 결과 ===")
print(result)

=== Few-shot 결과 ===
✅ 찬성 근거
- 출퇴근 시간 절감으로 업무 만족도와 집중도 향상
- 사무실 임대·운영 비용 절감
- 지역과 관계없이 우수 인재 채용 가능

❌ 반대 근거
- 소통·협업 저하로 팀워크 약화 가능
- 성과 관리와 업무 통제가 어려워질 수 있음
- 직무에 따라 보안·장비 문제 발생

💡 결론: 전면 도입보다는 직무 특성에 따라 선택적 재택근무를 적용하는 것이 현실적입니다.


---

## 3. Chain-of-Thought (CoT) Prompting

LLM에게 **단계별로 생각하도록** 유도합니다.  
복잡한 추론, 계획, 분석이 필요한 질문에 효과적입니다.

```
기본:  질문 → 바로 답변 (shallow)
CoT:  질문 → 단계1 → 단계2 → 단계3 → 결론 (deep)
```

In [5]:
# ── Chain-of-Thought: 단계별 사고 과정 명시 ───────────────────
cot_prompt = ChatPromptTemplate.from_messages([
    ("system", """질문에 답하기 전에 반드시 다음 4단계로 생각하고, 각 단계를 명시하세요.

[1단계: 문제 파악] 무엇을 묻고 있는가?
[2단계: 이해관계자 분석] 누가 영향을 받는가?
[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?
[4단계: 최종 판단] 근거를 바탕으로 한 결론은?"""),
    ("human", "{question}")
])

# 체인 생성 및 실행
chain = cot_prompt | llm | StrOutputParser()
result = chain.invoke({"question": "재택근무 전면 도입, 찬성해야 할까요?"})

print("=== Chain-of-Thought 결과 ===")
print(result)

=== Chain-of-Thought 결과 ===
[1단계: 문제 파악] 무엇을 묻고 있는가?  
재택근무를 **전면 도입할지**, 즉 회사 정책으로 **완전히 허용하는 것에 찬성해야 하는지**를 묻고 있습니다. 핵심은 생산성, 협업, 비용, 직원 만족도, 보안 등을 종합해 판단하는 것입니다.

[2단계: 이해관계자 분석] 누가 영향을 받는가?  
- **직원**: 출퇴근 부담 감소, 일·생활 균형 개선 가능  
- **관리자/팀장**: 성과관리, 소통, 팀 결속 유지 책임 증가  
- **회사/경영진**: 임대료·운영비 절감 또는 관리 복잡성 증가  
- **고객/협력사**: 응답 속도, 서비스 품질에 영향 가능  
- **신입/주니어 직원**: 교육·온보딩이 어려워질 수 있음  
- **IT/보안팀**: 원격접속, 정보보안, 장비 관리 부담 증가

[3단계: 장단점 검토] 도입/미도입 시 각각 어떤 결과가?  

**도입 시 장점**  
- 출퇴근 시간 절약으로 직원 만족도 상승  
- 지방 거주 인재 채용 등 인재풀 확대  
- 사무실 비용 절감 가능  
- 집중 업무에 유리한 경우가 있음

**도입 시 단점**  
- 팀 간 소통과 즉각적 협업이 약해질 수 있음  
- 업무 경계가 흐려져 번아웃 위험이 커질 수 있음  
- 성과평가가 어려워질 수 있음  
- 보안, 장비, 네트워크 문제 발생 가능  
- 신입 교육과 조직문화 형성이 약해질 수 있음

**미도입 시 장점**  
- 대면 협업과 즉시 피드백이 쉬움  
- 조직문화 유지와 신입 교육에 유리  
- 관리와 보안 통제가 상대적으로 쉬움

**미도입 시 단점**  
- 직원 피로와 이직률 증가 가능  
- 통근 시간과 비용 부담 큼  
- 채용 경쟁력 약화 가능  
- 유연성 부족으로 불만이 생길 수 있음

[4단계: 최


---

## 4. 역할(Role) 프롬프트

LLM에게 **특정 전문가 페르소나**를 부여합니다.  
같은 질문이라도 역할에 따라 완전히 다른 관점으로 답변합니다.

```
역할 없음:  중립적·일반적 답변
HR 전문가:  사람 중심의 조직 관점
CFO:       비용·효율 중심의 재무 관점
```

In [6]:
# ── 역할 프롬프트: 같은 질문, 다른 역할 ──────────────────────
def make_role_chain(role_description: str):
    """역할 설명을 받아 전문가 체인을 생성한다"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", role_description),
        ("human", "{question}")
    ])
    chain = prompt | llm | StrOutputParser()
    return chain


# 역할 1: HR 디렉터
hr_chain = make_role_chain("""
당신은 15년 경력의 HR 디렉터입니다.
직원 복지, 조직 문화, 인재 유지에 최우선 가치를 둡니다.
답변은 구성원 관점에서 3~4문장으로 핵심만 짚어주세요.
""")

# 역할 2: 재무이사(CFO)
cfo_chain = make_role_chain("""
당신은 10년 경력의 재무이사(CFO)입니다.
모든 결정을 비용-효익 분석으로 접근합니다.
답변은 수치 근거를 중심으로 3~4문장으로 핵심만 짚어주세요.
""")

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== HR 디렉터 관점 ===")
print(hr_chain.invoke({"question": question}))

print("\n=== CFO 관점 ===")
print(cfo_chain.invoke({"question": question}))

=== HR 디렉터 관점 ===
전면 재택근무는 구성원 입장에선 분명 큰 장점이 있지만, 모든 직무와 상황에 무조건 맞는 해법은 아닙니다. 출퇴근 부담이 줄고 일·생활 균형이 좋아지는 반면, 협업 속도나 소속감이 약해질 수 있어요. 그래서 저는 “전면 찬성/반대”보다, 직무별로 유연하게 적용하고 팀 단위 규칙을 함께 만드는 방향이 더 바람직하다고 봅니다. 구성원 만족과 몰입을 지키려면, 재택의 자유와 오프라인 협업의 장점을 함께 살리는 설계가 중요합니다.

=== CFO 관점 ===
재택근무 전면 도입은 **조건부 찬성**이 합리적입니다.  
보통 오피스 고정비가 인건비의 5~15% 수준이라면, 전면 재택으로 임대·관리비를 20~40% 절감해도 **총비용 절감 효과는 대개 2~8%** 정도에 그칩니다. 반면 이직률이 5~10%p만 악화돼도 채용·온보딩 비용과 생산성 손실로 절감분을 상당 부분 상쇄할 수 있습니다.  
따라서 **직무별로 생산성 측정이 가능하고, 협업 빈도가 낮은 기능직무부터** 도입하는 것이 ROI가 높습니다. 전면 도입은 비용절감보다 **인재 확보, 이직 감소, 출근시간 절감**이 더 큰 업종에서만 찬성하는 게 맞습니다.


---

## 5. 구조화된 출력 (Structured Output)

LLM의 출력을 **JSON 등 특정 형식으로 강제**합니다.  
후처리(파싱, DB 저장, API 응답)가 필요할 때 사용합니다.

```
일반 출력: 자연어 문장 → 파싱 불가
구조화:   JSON/YAML → 자동 파싱 가능 → 시스템 연동 가능
```

In [7]:
# ── 구조화된 출력: JSON 형식 강제 ────────────────────────────
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """반드시 아래 JSON 형식으로만 답변하세요. 다른 텍스트는 절대 포함하지 마세요.

  - 출력형식
  "topic": "주제",
  "pros": ["찬성 근거1", "찬성 근거2", "찬성 근거3"],
  "cons": ["반대 근거1", "반대 근거2"],
  "recommendation": "최종 권고사항",
  "confidence": 확신도(1~10)
"""),
    ("human", "다음 주제를 분석해줘: {question}")
])

# 체인 생성 및 실행
chain = structured_prompt | llm | StrOutputParser()
raw_output = chain.invoke({"question": "재택근무 전면 도입"})

print("=== LLM 원본 출력 ===")
print(raw_output)

# JSON 파싱
print("\n=== 파싱 후 활용 ===")
try:
    data = json.loads(raw_output)
    print(f"주제     : {data['topic']}")
    print(f"확신도   : {data['confidence']}/10")
    print(f"권고사항 : {data['recommendation']}")
    print(f"찬성 근거: {', '.join(data['pros'])}")
except json.JSONDecodeError as e:
    print(f"파싱 실패: {e}")
    print("→ 팁: 모델이 JSON을 못 지키면 출력에서 JSON 부분만 추출하거나 재시도 로직을 추가하세요.")

=== LLM 원본 출력 ===
{"topic":"재택근무 전면 도입","pros":["출퇴근 시간과 비용이 줄어들어 직원 만족도와 생산성이 향상될 수 있음","근무지 제약이 줄어 인재 채용 범위가 넓어지고 우수 인력 확보에 유리함","사무실 운영비, 임대료, 관리비 등 고정비를 절감할 수 있음"],"cons":["대면 협업이 줄어들어 소통 지연, 팀 결속 약화, 창의적 문제 해결이 어려워질 수 있음","업무 성과 관리와 보안 통제가 복잡해져 일부 직무에서는 효율이 떨어질 수 있음"],"recommendation":"전면 도입은 조직 문화와 업무 특성에 따라 장단점이 크므로, 모든 부서에 일괄 적용하기보다 하이브리드 방식이나 직무별 선택적 재택근무를 우선 검토하는 것이 바람직합니다. 원격 협업 체계, 성과관리 기준, 보안 정책이 충분히 갖춰진 조직이라면 점진적으로 확대 도입하는 것이 좋습니다.","confidence":8}

=== 파싱 후 활용 ===
주제     : 재택근무 전면 도입
확신도   : 8/10
권고사항 : 전면 도입은 조직 문화와 업무 특성에 따라 장단점이 크므로, 모든 부서에 일괄 적용하기보다 하이브리드 방식이나 직무별 선택적 재택근무를 우선 검토하는 것이 바람직합니다. 원격 협업 체계, 성과관리 기준, 보안 정책이 충분히 갖춰진 조직이라면 점진적으로 확대 도입하는 것이 좋습니다.
찬성 근거: 출퇴근 시간과 비용이 줄어들어 직원 만족도와 생산성이 향상될 수 있음, 근무지 제약이 줄어 인재 채용 범위가 넓어지고 우수 인력 확보에 유리함, 사무실 운영비, 임대료, 관리비 등 고정비를 절감할 수 있음


---

## 6. 프롬프트 템플릿 (재사용 가능한 팩토리)

**변수화된 프롬프트**를 만들어 다양한 도메인에 재사용합니다.  
같은 구조의 프롬프트를 반복해서 만들 때 효율적입니다.

```python
# 나쁜 예: 비슷한 프롬프트를 계속 복붙
legal_prompt   = "당신은 변호사입니다... {question}"
medical_prompt = "당신은 의사입니다... {question}"

# 좋은 예: 팩토리 함수로 생성
make_expert_chain(domain="법률", style="간결하게")
make_expert_chain(domain="의료", style="쉽게 설명")
```

In [8]:
# ── 프롬프트 템플릿 팩토리 ────────────────────────────────────
def make_analyst_chain(domain: str, perspective: str, output_format: str):
    """
    재사용 가능한 분석 체인을 생성한다.

    Args:
        domain      : 전문 도메인 (예: '경영', '기술', '심리')
        perspective : 분석 관점 (예: '비용 중심', '사람 중심')
        output_format: 출력 형식 지시문
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"""당신은 {domain} 분야 전문 분석가입니다.
{perspective} 관점에서 분석하세요.

출력 형식:
{output_format}"""),
        ("human", "{question}")
    ])
    
    # 체인 생성
    chain = prompt | llm | StrOutputParser()
    return chain


# 경영 분석 체인
biz_chain = make_analyst_chain(
    domain="경영",
    perspective="ROI와 생산성 중심",
    output_format="[핵심 지표] → [예상 효과] → [실행 권고] (각 2~3줄)"
)

# 조직심리 분석 체인
psych_chain = make_analyst_chain(
    domain="조직심리",
    perspective="구성원 동기·몰입 중심",
    output_format="[심리적 영향] → [리스크] → [권고사항] (각 2~3줄)"
)

question = "재택근무 전면 도입, 찬성해야 할까요?"

print("=== 경영 분석 관점 ===")
print(biz_chain.invoke({"question": question}))

print("\n=== 조직심리 관점 ===")
print(psych_chain.invoke({"question": question}))

=== 경영 분석 관점 ===
[핵심 지표] → 재택근무 전면 도입의 ROI는 **업무 성격에 따라 크게 갈립니다**. 반복적·문서 중심·개별 산출물 기반 업무는 생산성 유지/상승 가능성이 높지만, 협업·창의·대면 조율 비중이 높으면 조정 비용이 증가할 수 있습니다.  

[예상 효과] → 성공 시엔 **오피스 비용 절감, 채용 풀 확대, 이직률 감소**로 비용 대비 효과가 큽니다. 다만 **커뮤니케이션 손실, 관리자 통제비용, 온보딩 지연**이 커지면 생산성 하락으로 ROI가 역전될 수 있습니다.  

[실행 권고] → **전면 찬성보다 “부분 도입 후 성과 검증”이 더 합리적**입니다. 우선 직무별로 재택 적합도를 나누고, 8~12주 파일럿 후 **산출량·리드타임·이직률·직원당 매출/비용**으로 판단하는 것이 좋습니다.

=== 조직심리 관점 ===
[심리적 영향]  
재택근무 전면 도입은 자율성·통제감·일-생활 균형을 높여 구성원의 내재적 동기와 만족도를 끌어올릴 수 있습니다. 특히 집중 업무가 많은 사람에게는 몰입 경험이 강화될 가능성이 큽니다.

[리스크]  
반면 소속감 약화, 고립감, 비공식 학습 감소로 인해 조직몰입이 떨어질 수 있습니다. 또한 커뮤니케이션의 비대면화로 오해가 늘고, 성과가 보이는 사람 중심으로 평가가 기울어 공정성 인식이 흔들릴 수 있습니다.

[권고사항]  
전면 찬성 여부보다 “업무 특성 + 팀 상호의존도 + 구성원 선호” 기준으로 혼합형 정책을 권합니다. 전면 도입 시에는 정기적 대면 협업일, 명확한 성과기준, 고립감 점검과 팀 연결 장치를 함께 설계해야 합니다.


---

## 7. 비교 실험: 같은 질문, 다른 기법

지금까지 배운 6가지 기법을 **같은 질문**에 적용해 결과를 비교합니다.  
답변의 구조, 깊이, 형식이 얼마나 다른지 확인하세요.

In [9]:
# ── 비교 실험: 기법별 답변 길이 & 구조 비교 ──────────────────
QUESTION = "재택근무 전면 도입, 찬성해야 할까요?"

# 각 기법의 체인 목록
techniques = [
    ("Zero-shot",         zero_shot_prompt),
    ("Few-shot",          few_shot_prompt),
    ("Chain-of-Thought",  cot_prompt),
]

print(f"질문: {QUESTION}")
print("=" * 60)

results = {}
for name, prompt in techniques:
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"question": QUESTION})
    results[name] = answer

# 결과 요약 출력
print(f"\n{'기법':<20} {'글자수':>8} {'줄 수':>6}")
print("-" * 38)
for name, answer in results.items():
    print(f"{name:<20} {len(answer):>8,} {len(answer.splitlines()):>6}")

질문: 재택근무 전면 도입, 찬성해야 할까요?

기법                        글자수    줄 수
--------------------------------------
Zero-shot                 825     29
Few-shot                  218     11
Chain-of-Thought          841     32


In [10]:
# ── 답변 전체 출력 ────────────────────────────────────────────
for name, answer in results.items():
    print(f"\n{'='*60}")
    print(f"[{name}]")
    print('='*60)
    # 처음 400자만 출력 (너무 길면 ...으로 줄임)
    preview = answer[:400]
    print(preview)
    if len(answer) > 400:
        print(f"\n... (총 {len(answer):,}자 중 400자 미리보기)")


[Zero-shot]
한 줄로 답하면: **“전면 도입” 자체에는 신중해야 하지만, 조건이 맞으면 충분히 찬성할 만합니다.**  
즉, **무조건 찬성/반대보다 “부분 도입 + 직무별 설계”가 더 현실적**입니다.

## 찬성하는 이유
- **생산성 향상 가능**: 집중이 필요한 업무는 재택에서 더 효율적일 수 있음
- **우수 인재 확보**: 지역 제약이 줄어 채용 풀이 넓어짐
- **비용 절감**: 사무실 임대, 운영비, 출퇴근 비용 감소
- **일·생활 균형 개선**: 만족도와 이직률 개선에 도움
- **유연성 강화**: 돌봄, 건강 문제 등 개인 사정에 대응하기 쉬움

## 반대/우려되는 이유
- **협업과 소통 저하**: 즉각적인 피드백, 우연한 아이디어 교류가 줄 수 있음
- **성과 관리 어려움**: 결과보다 ‘보이

... (총 825자 중 400자 미리보기)

[Few-shot]
✅ 찬성 근거
- 출퇴근 시간 절감으로 직원 만족도 향상
- 지역 제한 없이 우수 인재 채용 가능
- 사무실 운영비 절감 효과 기대

❌ 반대 근거
- 소통·협업 저하로 업무 효율이 떨어질 수 있음
- 성과 관리와 보안 통제가 더 어려워짐
- 직무 특성상 대면 업무가 필요한 경우 적용이 어려움

💡 결론: 전면 도입보다는 직무별·부서별로 선택적 재택근무를 운영하는 방식이 더 현실적입니다.

[Chain-of-Thought]
[1단계: 문제 파악] 무엇을 묻고 있는가?  
재택근무를 **전면 도입해야 하는지**, 즉 회사/조직의 근무 방식을 **전면 원격으로 바꾸는 것에 찬성할지 여부**를 묻고 있습니다.

[2단계: 이해관계자 분석] 누가 영향을 받는가?  
- **직원**: 출퇴근 부담, 집중도, 일과 삶의 균형, 고립감에 영향  
- **경영진/관리자**: 생산성, 성과관리, 조직문화, 소통 방식에 영향  
- **인사/운영팀**: 채용, 평가, 복지, 보안, 제도 설계에 영향  
- **고객/협력사**: 응대 속도, 협업 품질, 서비스 안정성에 영향  
- **

---

## 핵심 정리

```python
# 프롬프트 엔지니어링 선택 가이드

# 1. Zero-shot: 빠른 테스트, 일반 질문
chain = ChatPromptTemplate.from_messages([("human", "{q}")]) | llm | StrOutputParser()

# 2. Few-shot: 특정 형식 유도, 스타일 일관성
#    → 시스템 프롬프트에 예시 2~3개 포함

# 3. Chain-of-Thought: 복잡한 추론, 다단계 분석
#    → "단계별로 생각하세요" 지시문 추가

# 4. 역할 프롬프트: 전문 도메인, 특정 관점 필요 시
#    → 구체적일수록 좋음 ("전문가" < "10년 경력 변호사")

# 5. 구조화 출력: 파싱·연동이 필요한 경우
#    → JSON 형식 + "다른 텍스트 포함 금지" 명시

# 6. 팩토리 패턴: 동일 구조 반복 시
def make_chain(domain, style):
    prompt = ChatPromptTemplate.from_messages([...])
    return prompt | llm | StrOutputParser()
```

| 기법 | 최적 상황 | 주의사항 |
|---|---|---|
| Zero-shot | 범용 질문, 빠른 프로토타입 | 형식 보장 없음 |
| Few-shot | 특정 형식·스타일 필요 | 예시가 너무 많으면 토큰 낭비 |
| CoT | 복잡한 추론·판단 | 응답이 길어짐 |
| 역할 | 전문 도메인 | 구체적일수록 효과적 |
| 구조화 출력 | API 연동, 파싱 필요 | 100% 보장 안 됨 (검증 로직 필요) |
| 팩토리 | 반복 사용 패턴 | 과도한 추상화 주의 |
